
# PASCAL VOC 2012 语义分割项目报告

## 一、数据集准备
### 1.1 数据集结构说明
```bash
下载VOC2012数据集后，解压得到：
VOCdevkit/
└── VOC2012/
    ├── JPEGImages/          # 存放所有原始图片（.jpg格式）
    ├── SegmentationClass/   # 存放所有分割标签（.png格式，彩色图）
    └── ImageSets/Segmentation/
        ├── train.txt        # 每行一个训练集图片名（不带后缀）
        └── val.txt          # 每行一个验证集图片名（不带后缀）
```

### 1.2 标签处理原理
- **为什么标签是彩色的？**  
  每个物体类别用特定RGB颜色标记（如汽车=灰色[128,128,128]）
  便于人工标注和检查
  
- **如何转换为训练用的单通道标签？**  
  ```python
  # 示例：将彩色标签转为0-20的索引值
  label = np.array(Image.open("标签路径.png"))  # 读取为RGB数组
  index_map = np.zeros(label.shape[:2])       # 创建空矩阵
  
  # 遍历所有定义的颜色
  for class_idx, color in enumerate(VOC_COLORMAP):
      # 找到所有是该颜色的像素位置
      mask = np.all(label == color, axis=2) 
      index_map[mask] = class_idx  # 设为类别索引
  ```

## 二、代码关键实现
### 2.1 数据加载流程
1. **读取文件列表**  
   ```python
   # 从val.txt读取验证集图片名（如2007_000033）
   with open("VOC2012/ImageSets/Segmentation/val.txt") as f:
       image_ids = [line.strip() for line in f]
   ```

2. **配对图像和标签**  
   ```python
   # 图像路径：JPEGImages/2007_000033.jpg
   image_path = f"VOC2012/JPEGImages/{image_ids[0]}.jpg"
   
   # 标签路径：SegmentationClass/2007_000033.png  
   label_path = f"VOC2012/SegmentationClass/{image_ids[0]}.png"
   ```

3. **同步数据增强**  
   ```python
   # 保证图像和标签的变换完全一致
   def transform(img, label):
       # 随机缩放（相同参数）
       scale = random.uniform(0.5, 2.0)
       new_size = (int(img.width*scale), int(img.height*scale))
       
       img = img.resize(new_size, Image.BILINEAR)   # 图像用双线性插值
       label = label.resize(new_size, Image.NEAREST) # 标签用最近邻（避免混色）
       
       # 随机裁剪（相同位置）
       i, j, h, w = RandomCrop.get_params(img, (512, 512))
       img = transforms.functional.crop(img, i, j, h, w)
       label = transforms.functional.crop(label, i, j, h, w)
       
       return img, label
   ```

4. **转换为单通道**
    ```python
    def _encode_label(self, label_img):
    """ 
    RGB彩色标签图 ——> 单通道的类别索引图
    输入:
        label_img: PIL.Image对象，每个像素是VOC预定义的21种颜色之一
    输出:
        torch.LongTensor:           
        - 数值0-20 对应21个类别（如0=背景，2=自行车）
        - 255表示物体边界（不参与训练）
    """
    # 步骤1：将PIL图像转为numpy数组，shape为[H,W,3]（RGB通道）
    label = np.array(label_img)
    
    # 步骤2：创建全零矩阵，用于存储类别索引，shape[H,W]
    index_map = np.zeros(label.shape[:2], dtype=np.int64)
    
    # 步骤3：处理特殊边界区域（VOC数据集中边界用[224,224,192]标记）
    # np.all(axis=2)表示在RGB通道维度上判断是否完全匹配
    boundary_mask = np.all(label == [224, 224, 192], axis=2)
    index_map[boundary_mask] = 255  # 边界设为255（训练时通常忽略）
    
    # 步骤4：遍历VOC颜色映射表，将RGB值映射为类别索引
    for idx, color in enumerate(self.VOC_COLORMAP):
        # 生成当前类别的布尔值（True表示该像素属于当前类别）
        class_mask = np.all(label == np.array(color).reshape(1,1,3), axis=2)
        
        index_map[class_mask] = idx
        
        """ 
        颜色匹配原理：
        假设当前处理"汽车"类别（索引7，颜色[128,128,128]）
        np.all(label == [128,128,128], axis=2) 会找到所有RGB值为灰色的像素
        然后将这些像素位置在index_map中设为7
        """
    
    # 步骤5：将numpy数组转为PyTorch张量
    return torch.from_numpy(index_map).long()

    """
    最终得到的index_map示例：
    [[0 0 0 ... 7 7 7],
     [0 0 0 ... 7 7 7],
     ...,
     [15 15 15 ... 255 255 255]]
    其中：
    - 0 代表背景
    - 7 代表汽车
    - 15 代表人
    - 255 代表边界（不参与训练）
    """
    ```
```mermaid
flowchart TD
    A[开始] --> B[加载数据集]
    B --> C["读取val.txt/train.txt"]
    C --> D["构建图像路径列表\nJPEGImages/xxx.jpg"]
    C --> E["构建标签路径列表\nSegmentationClass/xxx.png"]
    
    D --> F[读取图像]
    E --> F
    F --> G["同步变换\n(缩放+裁剪+翻转)"]
    G --> H["图像处理\n1. 归一化\n2. ToTensor"]
    G --> I["标签处理\n1. RGB转索引\n2. 边界标记"]
    
    H --> J[训练/推理]
    I --> J
```



### 2.2 模型推理
```python
# 1. 加载预训练模型
model = deeplabv3_resnet101(weights="DEFAULT")

# 2. 图像预处理
transform = Compose([
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
image = transform(Image.open("图片路径.jpg"))

# 3. 执行预测
with torch.no_grad():
    output = model(image.unsqueeze(0))['out']  # 添加batch维度
    pred = output.argmax(dim=1).squeeze().cpu().numpy()  # 获取预测类别
```

## 三、结果分析（含示例）
### 3.1 简单样本结果
| 样本 | 预测效果描述 | 可视化 |
|------|--------------|--------|
| 样本0 | 飞机主体可识别，边缘附近有丢失 | ![sample0](results/simple/sample_0.png) |
| 样本15 | 自行车外轮廓识别准确，内部细节缺失 | ![sample15](results/simple/sample_15.png) |
| 样本20 | 识别准确 | ![sample20](results/simple/sample_20.png) |

### 3.2 复杂场景问题
**样本2007_001311分析**：  
- **主要错误**：
  - 自行车轮漏检（红色区域）  
    ![bicycle_error](results/complex/scene_analysis/bicycle_errors.png)
  - 人物阴影误判（蓝色区域）  
    ![person_error](results/complex/scene_analysis/person_errors.png)

- **量化指标**： 

| **类别**      | **精确度 (Precision)** | **召回率 (Recall)** | **IoU**  | **分析**                     |
|--------------|----------------------|-------------------|---------|-----------------------------|
| **background** | 94.84%              | 96.79%            | 91.95%  | 背景识别准确，误检/漏检极少          |
| **person**    | 84.05%              | 91.26%            | 77.78%  | 检测全面，但存在少量类别误判   |
| **bicycle**   | 45.46%              | 93.76%            | 44.12%  | 漏检少，但易将其他物体判为自行车 |

**主要问题**：  
自行车类误检率高，需针对性优化模型对细小/复杂结构的识别能力。自行车镂空的部分被填成实心。
   
## 四、优化
### 4.1 改善边界精度
```python
# 添加条件随机场后处理
import pydensecrf.densecrf as dcrf

def crf_refine(image, pred):
    # 根据颜色和位置优化边界
    d = dcrf.DenseCRF2D(image.shape[1], image.shape[0], 21)
    ...
    return refined_pred
```
